In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
load_dotenv()
if os.environ['GROQ_API_KEY']:
    print("API key available")
llm=ChatGroq(model="llama-3.3-70b-versatile",temperature=0.7)  

d:\Shivansh Thakur\rAg2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


API key available


## Rag on PDF

#### Extract text from pdf

In [5]:
from langchain_community.document_loaders import PyPDFLoader
pdf_path = "Docs/FINAL REPORT.pdf"
loader =PyPDFLoader(pdf_path)
docs=loader.load()
docs

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-04-13T10:25:43+00:00', 'author': 'Vartika Mathur', 'moddate': '2026-04-13T10:25:44+00:00', 'source': 'Docs/FINAL REPORT.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1'}, page_content='Semester VII  \n    Hybrid Deep Learning Model for Sector Rotation in \nthe IndIan Stock Market…  \n  \n \n                                                                               Submitted by:- \n                                        AYUSHMAN SHUKLA \n \n \n \n \n                                                         SUPERVISOR:- DR. J. LALITA \n                                  CO- SUPERVISOR:- \n \n \n \n                                         Sri Venkateswara College \nUniversity of Delhi  \nBenito Juarez Road, Dhaula Kuan  \nNEW DELHI-110057 \n \nSRI VENKATESWARA COLLEGE \nNEP: UGCF 2022 \n \nDissertation/Academic Project/ \nEntrepreneurship Report \n2025-26'),
 Document(metad

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter =RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)
chunks=splitter.split_documents(docs)
chunks

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-04-13T10:25:43+00:00', 'author': 'Vartika Mathur', 'moddate': '2026-04-13T10:25:44+00:00', 'source': 'Docs/FINAL REPORT.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1'}, page_content='Semester VII  \n    Hybrid Deep Learning Model for Sector Rotation in \nthe IndIan Stock Market…  \n  \n \n                                                                               Submitted by:- \n                                        AYUSHMAN SHUKLA \n \n \n \n \n                                                         SUPERVISOR:- DR. J. LALITA \n                                  CO- SUPERVISOR:- \n \n \n \n                                         Sri Venkateswara College \nUniversity of Delhi  \nBenito Juarez Road, Dhaula Kuan  \nNEW DELHI-110057 \n \nSRI VENKATESWARA COLLEGE \nNEP: UGCF 2022 \n \nDissertation/Academic Project/ \nEntrepreneurship Report \n2025-26'),
 Document(metad

In [12]:
len(chunks)

50

#### Vector Store


In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

from langchain_community.vectorstores import Chroma
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./Vector"
)

d:\Shivansh Thakur\rAg2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7062.13it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NameError: name 'chunks' is not defined

In [19]:
vectorstore.similarity_search("what is this document about?")


[Document(metadata={'source': 'Docs/FINAL REPORT.pdf', 'moddate': '2026-04-13T10:25:44+00:00', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-04-13T10:25:43+00:00', 'page_label': '3', 'author': 'Vartika Mathur', 'page': 2, 'total_pages': 23, 'producer': 'www.ilovepdf.com'}, page_content='This acknowledgement reflects my gratitude during the ongoing phase of the project, \nand I look forward to continuing my work with the same dedication. \n \nWith sincere appreciation, \nAYUSHMAN SHUKLA'),
 Document(metadata={'total_pages': 23, 'creator': 'Microsoft® Word 2016', 'author': 'Vartika Mathur', 'page_label': '3', 'moddate': '2026-04-13T10:25:44+00:00', 'creationdate': '2026-04-13T10:25:43+00:00', 'producer': 'www.ilovepdf.com', 'page': 2, 'source': 'Docs/FINAL REPORT.pdf'}, page_content='This acknowledgement reflects my gratitude during the ongoing phase of the project, \nand I look forward to continuing my work with the same dedication. \n \nWith sincere appreciation, \nAYUSHMAN 

#### Reuse the vector DB

In [2]:
vectorstore_persist=Chroma(
    persist_directory="./Vector/",
    embedding_function=embedding_model
)

C:\Users\shthakur\AppData\Local\Temp\ipykernel_12648\2383861653.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore_persist=Chroma(


In [3]:
vectorstore_persist.similarity_search("what is this document about?")

[Document(metadata={'source': 'Docs/FINAL REPORT.pdf', 'page_label': '3', 'total_pages': 23, 'author': 'Vartika Mathur', 'creator': 'Microsoft® Word 2016', 'moddate': '2026-04-13T10:25:44+00:00', 'producer': 'www.ilovepdf.com', 'page': 2, 'creationdate': '2026-04-13T10:25:43+00:00'}, page_content='This acknowledgement reflects my gratitude during the ongoing phase of the project, \nand I look forward to continuing my work with the same dedication. \n \nWith sincere appreciation, \nAYUSHMAN SHUKLA'),
 Document(metadata={'page': 21, 'moddate': '2026-04-13T10:25:44+00:00', 'producer': 'www.ilovepdf.com', 'author': 'Vartika Mathur', 'creationdate': '2026-04-13T10:25:43+00:00', 'total_pages': 23, 'page_label': '22', 'creator': 'Microsoft® Word 2016', 'source': 'Docs/FINAL REPORT.pdf'}, page_content='https://finance.yahoo.com/ \n[12] TensorFlow Developers, “TensorFlow documentation,” 2024. [Online]. Available: \nhttps://www.tensorflow.org/ \n[13] Keras Documentation, “Keras deep learning API